In [ ]:
from pathlib import Path
from loguru import logger


class ProjectPath:
    @staticmethod
    def find_project_root(marker="pyproject.toml"):
        path = Path.cwd().resolve()
        while path != path.parent:
            if (path / marker).exists():
                return path
            path = path.parent
        raise FileNotFoundError(f"Could not find project root with marker: {marker}")

    ROOT = find_project_root()
    logger.info(f"Project root directory: {ROOT.resolve()}")
    assert ROOT.exists()


PROJECT_DIR = ProjectPath().ROOT

## Purpose of This Notebook

This notebook loads and parses log file data for the `RPE` experiments into a database.

The primary objectives are:


> To load the log files determined to be included in the analysis in `notebooks/logfile-reconciliation-RPE.ipynb`.
> To parse and extract the data for each experiment from each corresponding log file and push this data into a [DuckDB](https://duckdb.org) database.

The notebook can be "rendered" using the very powerful [Quarto](https://quarto.org) publication tool to produce HTML (webpage) output from the notebook. The tags `# | echo: false` suppress the output of the Python code in a cell and only display the results (output from each cell.)
See https://quarto.org/docs/reference/formats/html.html#execution for details. 

In [ ]:
# | echo: false

from pprint import pprint

import duckdb
import pandas as pd
from horse_logic.logfiles import Event, Experiment, Logfile, Logs, Trial
from horse_logic.utils import (
    create_tables_from_sql_file,
    display_class_definition,
    export_data_to_csv,
    set_custom_logger_format,
)
from IPython.display import Markdown, display

In [ ]:
# | echo: false

import itables.options as opt
from itables import init_notebook_mode

init_notebook_mode(
    all_interactive=True, connected=False
)  # Display Pandas dataframes in a more friendly paginated manner
opt.pageLength = 20  # Display 20 rows per page

In [ ]:
# | echo: false

set_custom_logger_format()

### Load Subject information

In [ ]:
# | echo: false


def get_subject_info():
    HORSE_ORDER_XLSX = "Cohort data for MB.xlsx"
    HORSE_ORDER_PATH = PROJECT_DIR / "docs" / "provided"
    HORSE_ORDER_FILEPATH = HORSE_ORDER_PATH / HORSE_ORDER_XLSX

    if HORSE_ORDER_FILEPATH.exists():
        subject_df = pd.read_excel(HORSE_ORDER_FILEPATH)
        logger.info(f"Loaded horse order info from: {HORSE_ORDER_FILEPATH}")

        subject_df.rename(
            {"No": "subject_number", "Horse": "subject_name"}, axis=1, inplace=True
        )  # Rename columns
        subject_df["subject_name"] = subject_df[
            "subject_name"
        ].str.lower()  # Ensure lower case names for later subject lookup
        return subject_df
    else:
        logger.error(f"Horse order info not found: {HORSE_ORDER_FILEPATH}")
        return None

In [ ]:
subject_df = get_subject_info()

### Define directory information: Inputs (log files), data and output directories

In [ ]:
# | echo: false

LOGFILES_DIR = PROJECT_DIR / "data/results/zips/data_17Jan2020_email_hillydale_equine"

assert LOGFILES_DIR.exists()
logger.info(f"Logfiles dir: {LOGFILES_DIR.resolve()}")

In [ ]:
# | echo: false


DATA_DIR = PROJECT_DIR / "data"
SQL_DIR = PROJECT_DIR / "sql"
EXPERIMENT_TYPE = "RPE"  # or CB

assert DATA_DIR.exists()

DATA_DB = DATA_DIR / f"Experiments_{EXPERIMENT_TYPE}_2023_Q4.ddb"  # DuckDB database name
db_exists = DATA_DB.exists()

logger.info(f"Database file: {DATA_DB.resolve()}")

In [ ]:
# | echo: false

OUTPUT_DIR = DATA_DIR / f"results/{EXPERIMENT_TYPE}"

assert OUTPUT_DIR.exists()
logger.info(f"Outputs dir: {OUTPUT_DIR.resolve()}")

### Load logfiles from CSV list of included files

As determined in `logfiles-reconciliation.ipynb`

In [ ]:
# | echo: false

logs = Logs(path=LOGFILES_DIR)
logs.load_specific_logfiles_from_csv(OUTPUT_DIR / "all_included_files.csv")

In [ ]:
# | echo: false

logger.info(logs)

### Explore `regex` parsing of a single example log file

Analysis code is in `logfiles.py`. Explore parsing log information for specific example defined below.

In [ ]:
# | echo: false

logfile_eg = f"{logs.path}/Experiment_2023-11-17T05:11:53.655051_clover_nan_RPE-R.log"

In [ ]:
# | echo: false

log_eg = Logfile(logfile_eg)
logger.info(f"Example log file for regex parsing: {log_eg.file_name}")

### Data structures

These are created as `dataclasses` in Python (in `logfiles.py`) and the corresponding TABLES in SQL (`sql/create_experiment_tables_ddb.sql`).

#### `Experiment` class

In [ ]:
# | echo: false

display_class_definition(Experiment)

#### e.g. Parse experiment details and comment from log file

In [ ]:
# | echo: false

pprint(log_eg.parse_filename_components())

In [ ]:
# | echo: false

log_eg.parse_comments()

#### `Parameter` class

Parameters are stored a a `Dict` in the Experiment class (see above). These get mapped to a `JSON` field in the database.

##### e.g. Parse Parameters from log file

In [ ]:
# | echo: false

log_eg.parse_parameters()

#### `Trial` and `Event` classes

In [ ]:
# | echo: false

display_class_definition(Trial)

In [ ]:
# | echo: false

display_class_definition(Event)

#### e.g. Parsing of trials and events info from log file

In [ ]:
# | echo: false

n_trial = len(log_eg.parse_trials_and_events()[0])

In [ ]:
# | echo: false

trial_event_df = pd.DataFrame(log_eg.parse_trials_and_events()[1])

In [ ]:
# | echo: false

trial_event_df

##### Example of Trials and Event information

Example of Events for each trial number

In [ ]:
# | echo: false


def display_events_by_trial_number(trial_event_df):
    n_trial = trial_event_df["TrialNumber"].max()
    for i_trial in range(n_trial):
        trial_event_df.loc[:, "time_diff"] = pd.to_datetime(trial_event_df["EventDateTime"]).diff()
        display(Markdown(f"### Trial #{i_trial+1}"))
        display(trial_event_df[trial_event_df["TrialNumber"] == i_trial + 1])


display_events_by_trial_number(trial_event_df)

### Bringing this all together and putting the data in DuckDB database

In [ ]:
# | echo: false


def create_replace_ddb():
    if db_exists:  # Remove database file if it exists
        logger.info(f"Deleted existing database file: {DATA_DB}")
        DATA_DB.unlink()
    con = duckdb.connect(database=str(DATA_DB))
    create_tables_from_sql_file(con, SQL_DIR / "create_experiment_tables_ddb.sql")
    return con


con = create_replace_ddb()

In [ ]:
# | echo: false

# Log files excluded as they cannot be successfully parsed - indicating poor quality / bad data

extra_excludes = sorted(
    [
        "Experiment_2023-10-03T14:14:32.928978_bonnie_10_RPE-A.log",
        "Experiment_2023-10-02T21:28:43.621970_bonnie_2_RPE-R.log",
        "Experiment_2023-10-03T14:55:06.781765_bonnie_15_RPE-ER.log",
        "Experiment_2023-10-03T15:46:19.429509_bonnie_16_RPE-ER.log",
        "Experiment_2023-10-04T11:15:57.929926_bonnie_36_RPE-A.log",
        "Experiment_2023-10-12T10:21:52.390998_bonnie_61_RPE-A.log",
        "Experiment_2023-10-16T16:54:06.481712_bonnie_67_RPE-A.log",
        "Experiment_2023-10-18T09:35:29.804367_atom_3_RPE-H.log",
        "Experiment_2023-10-19T14:37:29.776913_bonnie_71_RPE-H.log",
        "Experiment_2023-10-20T11:31:25.005165_mowgli_7_RPE-E.log",
        "Experiment_2023-11-15T01:23:45.430018_jelly_1.0_RPE-A.log",
        "Experiment_2023-11-15T16:43:09.229990_jelly_6.0_RPE-H.log",
    ]
)


export_data_to_csv(
    pd.DataFrame(extra_excludes),
    "Additional log files excluded as bad data",
    f"{OUTPUT_DIR / 'extra_excludes.csv'}",
)

**TODO (CH):** Should record why these extra log files were excluded and ideally move back into the reconciliation notebook. From memory these could not be parsed for some reason.

In [ ]:
# | echo: false

logfiles_to_process = sorted(
    [logfile.file_name for logfile in logs.logfiles if logfile.file_name not in extra_excludes]
)

export_data_to_csv(
    pd.DataFrame(sorted(logfiles_to_process)),
    "List of logfiles to be loaded to database",
    f"{OUTPUT_DIR / 'logfiles-to-database.csv'}",
)

In [ ]:
# | echo: false


def parse_and_load_logfiles_to_database(con, logs, logfiles_to_process):

    # Initialise the unique (integer) id for each Trial (TrialID) and each Event (EventID)
    trial_id = 0
    event_id = 0

    # Loop over Experiments

    for exp_id, log_filename in enumerate(
        logfiles_to_process
    ):  # Loop over Experiments - starting at exp_id = 0 (ExperimentID)
        logger.info(f"{exp_id}: {logs.path}/{log_filename}")
        logfile = Logfile(f"{logs.path}/{log_filename}")
        exp_details = logfile.parse_filename_components()
        exp_details.ExperimentID = exp_id
        exp_details.Cohort = subject_df.loc[
            subject_df["subject_name"] == exp_details.SubjectName, "Cohort"
        ].iloc[0]
        exp_details.SubjectNumber = int(
            subject_df.loc[subject_df["subject_name"] == exp_details.SubjectName, "subject_number"].iloc[
                0
            ]
        )
        exp_details.Parameters = logfile.parse_parameters()
        logfile.insert_record_to_database(con, exp_details)  # Add each Experiment

        # Loop over Trials

        trial_events_df = pd.DataFrame(logfile.parse_trials_and_events()[1])
        n_trial = trial_events_df["TrialNumber"].max()

        for trial_number in range(n_trial):
            trial_events_df.loc[:, "EventDateTime"] = pd.to_datetime(trial_events_df["EventDateTime"])
            trial_events_df.loc[:, "time_diff"] = trial_events_df["EventDateTime"].diff()

            trial_start_time = trial_events_df.iloc[0][
                "EventDateTime"
            ]  # Start tone of the first event in the trial
            trial_end_time = trial_events_df.iloc[-1]["EventDateTime"]  # Last event time in the trial

            logfile.insert_record_to_database(
                con, Trial(trial_id, exp_id, trial_number + 1, trial_start_time, trial_end_time)
            )  # Add each Trial

            # Loop over Events

            events_df = trial_events_df[trial_events_df["TrialNumber"] == trial_number + 1]
            n_event = len(events_df)

            for event_number in range(n_event):
                event_time = events_df.iloc[event_number]["EventDateTime"]
                event_type = events_df.iloc[event_number]["EventType"]
                event_elapsed_time = (event_time - trial_start_time).seconds
                logfile.insert_record_to_database(
                    con,
                    Event(
                        event_id, trial_id, event_number + 1, event_time, event_type, event_elapsed_time
                    ),
                )  # Add each Event
                event_id += 1
            trial_id += 1

    return len(logfiles_to_process), trial_id, event_id

In [ ]:
# | echo: false

n_experiment, n_trial, n_event = parse_and_load_logfiles_to_database(con, logs, logfiles_to_process)

In [ ]:
# | echo: false

logger.info(f"# Loaded - Experiments: {n_experiment}, Trials: {n_trial}, Events: {n_event}")

#### Run some basic SQL queries to examine the overall characteristics of the data

The companion notebook `database-queries.ipynb` explores the database in more detail.

In [ ]:
# Cross-check queries

con.sql("SHOW TABLES;")

#### Experiments table

In [ ]:
con.sql("SELECT COUNT(*) as n_experiment FROM Experiments")

In [ ]:
con.sql("SELECT DISTINCT ExperimentType FROM Experiments")

In [ ]:
con.sql("SELECT DISTINCT SubjectName FROM Experiments ORDER BY SubjectName")

In [ ]:
con.sql("SELECT DISTINCT Cohort FROM Experiments")

In [ ]:
# Cross-check against inputs - OK

con.sql(
    "SELECT DISTINCT Cohort, SubjectNumber, SubjectName FROM Experiments ORDER BY SubjectNumber"
)  # .df().to_csv(HORSE_ORDER_PATH / "cross_check_subjects.csv")

#### Trials table

In [ ]:
con.sql("SELECT COUNT(*) as n_trial FROM Trials")

#### Events table

In [ ]:
con.sql("SELECT COUNT(*) as n_event FROM Events")

In [ ]:
con.sql("SELECT COUNT (DISTINCT EventType) as n_event_type FROM Events")

#### Parameters table

In [ ]:
con.sql(
    """
    SELECT COUNT (DISTINCT Parameters) 
    FROM
        Experiments
"""
)

In [ ]:
# TODO - Count elements in the JSON ()
con.sql(
    """
    SELECT DISTINCT json_keys(Parameters) as num_items, ExperimentType FROM Experiments
        """
)

In [ ]:
# SQL to retrieve specific parameter (INITIAL_DELAY) from specific ExperimentID

con.sql(
    """
    SELECT 
        ExperimentID,
        JSON_EXTRACT(Parameters, '$.INITIAL_DELAY') AS InitialDelay
    FROM 
        Experiments
    WHERE
        ExperimentID = 42
"""
)

In [ ]:
con.sql(
    """
    SELECT DISTINCT Parameters 
    FROM
        Experiments
        """
).df()

In [ ]:
parameter_df = con.sql("SELECT Parameters FROM Experiments").df()

In [ ]:
parameter_df["Parameters"].nunique()

In [ ]:
parameter_df["Parameters"].unique()

In [ ]:
con.sql(
    """
SELECT 
    *
FROM 
    Experiments
WHERE
    JSON_EXTRACT(Parameters, '$.MODE') = '"test"'
"""
)

In [ ]:
def get_parameter_for_experimentid(parameter, id):
    return con.sql(
        f"""
    SELECT 
        JSON_EXTRACT(Parameters, '$.{parameter}') AS InitialDelay
    FROM 
        Experiments
    WHERE
        ExperimentID = {id}
"""
    )

In [ ]:
get_parameter_for_experimentid("SERVO_MODE", 42)

In [ ]:
get_parameter_for_experimentid("MODE", 0)

In [ ]:
def export_database_tables(con, output_dir, output_format="xlsx"):
    # Get the list of tables in the database
    try:
        tables = con.sql("SHOW TABLES;").fetchall()

        if output_format == "xlsx":
            # Write to one Excel file by sheet name
            with pd.ExcelWriter(f"{output_dir}/all_tables.xlsx") as writer:
                for table in tables:
                    table_name = table[0]
                    # Query the table and convert it to a pandas DataFrame
                    df = con.table(table_name).to_df()
                    df.to_excel(writer, sheet_name=table_name, index=False)
            logger.info(f"Exported database tables to: {output_dir}/all_tables.xlsx")

        elif output_format in ["csv", "parquet"]:
            for table in tables:
                table_name = table[0]
                # Query the table and convert it to a pandas DataFrame
                df = con.table(table_name).to_df()

                if output_format == "csv":
                    # Write to a CSV file
                    df.to_csv(f"{output_dir}/{table_name}.csv", index=False)
                    logger.info(f"Exported database table to: {output_dir}/{table_name}.csv")
                elif output_format == "parquet":
                    # Write to a Parquet file
                    df.to_parquet(f"{output_dir}/{table_name}.parquet", index=False)
                    logger.info(f"Exported database table to: {output_dir}/{table_name}.parquet")

        else:
            raise ValueError(f"Unsupported output format: {output_format}")
    except Exception as e:
        logger.error(f"{e} - Database connection not available.")

In [ ]:
export_database_tables(con, OUTPUT_DIR / "export_tables", output_format="xlsx")

In [ ]:
export_database_tables(con, OUTPUT_DIR / "export_tables", output_format="csv")

In [ ]:
con.close()